# 5 – RAG with RAPTOR

Examples 1–4 all retrieve *chunks*. That works when the answer sits in one place. Many real questions do not:

- "What debt and lease obligations does the company have?" – spread over four notes and two tables.
- "What kind of company is this, and how did 2019 compare with 2018?" – the answer is the whole document.

Top-*k* chunk retrieval returns *k* fragments of such an answer and misses the rest, and the LLM answers from an incomplete picture. Raising *k* does not fix it: the chunks you need are not necessarily the most similar ones, and at some point the context is full of noise.

The "summaries and trees" family of techniques attacks this at indexing time:

| Technique | Idea |
|---|---|
| **Index the summary** | Embed an LLM summary of each document or table, but return the full text when it matches. Good for tables and long reports. |
| **RAPTOR** | Cluster the chunks by embedding, summarise each cluster, embed the summaries, cluster again. A tree whose top speaks about the whole corpus (Sarthi et al., 2024, [arXiv:2401.18059](https://arxiv.org/abs/2401.18059)). |
| **Retrieve at any level** | Detail questions hit the leaves; "what is this report about" hits a node higher up. The first step towards hierarchy, which graph methods pick up again. |

This notebook builds a RAPTOR tree over one document and compares three ways of retrieving from it.

```
                        ┌──────────── root ────────────┐        layer 3: the whole document
                        │                              │
              ┌──── summary ────┐                  summary       layer 2
              │        │        │                     │
          summary   summary   summary   ...        summary       layer 1: one per cluster
          /  |  \    /  |  \    /  \               /  |  \
        leaf leaf leaf leaf leaf leaf leaf  ...  leaf leaf leaf  layer 0: the chunks
```

**The document**: `data/document_a.md`, the 2019 financial statements of a small professional employer organization – four statements as HTML tables and ten notes, about 20,000 characters. It is a realistic input: headings, tables, numbers, boilerplate.

As in Examples 1–3, the LLM and the embedding model are served by Ollama, and Chroma is the vector store.

## How RAPTOR builds and uses the tree

### Building the tree (indexing time)

1. **Leaves.** Chunk the document as usual. (The paper uses ~100-token chunks that respect sentence boundaries; we use structure-aware chunks, see below.)
2. **Embed** every node of the current layer.
3. **Cluster** the nodes by their vectors. The paper reduces the dimensionality with UMAP, fits a Gaussian Mixture Model (GMM) and picks the number of clusters with the Bayesian Information Criterion (BIC). The clustering is *soft*: a node belongs to every cluster for which its membership probability exceeds a threshold, so a chunk that is about both debt and leases can feed two summaries. We keep GMM + BIC + soft membership and swap UMAP for PCA: it is deterministic, has no parameters, and at the 8–10 dimensions used for clustering it keeps the neighbourhoods of our leaves better than UMAP does. The companion notebooks **5a** (PCA and UMAP), **5b** (Gaussian mixtures) and **5c** (BIC) explain these three building blocks in detail, with visualisations.
4. **Summarise** each cluster with an LLM. Every summary becomes a node of the next layer and remembers its children.
5. **Repeat** steps 2–4 on the new layer until the layer is too small to cluster, or the GMM finds that one cluster fits best. A final root summary tops the tree.

The cost is one LLM call per cluster per layer – a dozen for our document, around ten minutes with a 24B model on a laptop. Like Example 2, we cache the result on disk.

### Retrieving from the tree (query time)

The paper describes two strategies:

- **Tree traversal** – start at the root, keep the top-*k* nodes by cosine similarity, descend into their children, repeat down to the leaves. The result is a path through the tree: some context from every level.
- **Collapsed tree** – ignore the structure. Put *all* nodes of *all* layers into one index and run a plain similarity search. The query decides which granularity wins: a question about one interest rate is closest to a leaf, a question about the company's year is closest to a summary.

The paper found the collapsed tree to be both simpler and more accurate, so that is what we feed the LLM; traversal is implemented for comparison.

### Why it helps

- Summaries are *synthesised* text. A layer-1 node about debt pulls together numbers from four notes, so a question spanning them matches one node instead of needing four lucky hits.
- Broad questions get broad nodes, detail questions still get leaves. Nothing is lost: the leaves stay in the index.
- Compared with Example 2: both spend LLM calls at indexing time, but propositions go *finer* than the chunks and RAPTOR goes *coarser*. The two are complementary.

Caveats: a summary can drop or distort a detail, so the summarisation prompt must insist on keeping figures; cluster quality depends on the embedding model and on how many nodes there are; the tree has to be rebuilt when the documents change.

## Imports

New compared with the earlier notebooks:

- `numpy` – to turn the lists of embeddings into a matrix and to read off cluster memberships.
- `sklearn.decomposition.PCA` and `sklearn.mixture.GaussianMixture` – the dimensionality reduction and the clustering model. scikit-learn is already installed as a dependency of `sentence-transformers`.
- `MarkdownHeaderTextSplitter` – a structure-aware splitter that cuts a Markdown file at its headings and records the heading path of every piece.
- `json` – for the tree cache.

Everything else (Ollama LLM and embeddings, Chroma, `Document`, prompt template, output parser, recursive splitter) is familiar from Examples 1–3.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

## Configuration

- `DATA_FILE` – the one document we build the tree over.
- `CACHE_FILE` – where the finished tree (all nodes with their texts and children) is saved. Delete it to rebuild, for example after changing the chunking, the summary prompt or the LLM.
- `LLM_MODEL`, `EMBEDDING_MODEL` – the Ollama-served models from Examples 1–3.
- `CHUNK_SIZE = 1200` characters – larger than in the earlier examples, because the leaves have to hold table rows, and a table row cut in half is useless. `CHUNK_OVERLAP = 0` as in Example 4.
- `MIN_CHARS = 80` – pieces shorter than this are dropped. In this document they are date lines ("December 31, 2019 and 2018") and the sentence "The accompanying notes are an integral part of these financial statements", which carry no retrievable fact and would otherwise form a cluster of their own.
- `REDUCED_DIM = 8`, `MAX_CLUSTERS = 10` – the PCA dimension and the largest number of clusters the BIC search considers. Both are additionally capped by the size of the layer (see `cluster_nodes`), so small upper layers are not over-split.
- `CLUSTER_THRESHOLD = 0.1` – a node joins every cluster whose membership probability exceeds this value (the paper's value).
- `MAX_CLUSTER_CHARS = 5000` – a cluster whose texts add up to more than this (roughly 1,300 tokens) is cut in half, in document order, until the parts fit. A summary of a bounded cluster compresses less and stays faithful; it is also the only reliable way to keep a local model's summaries short, as we will see.
- `MIN_NODES = 4` – stop clustering when a layer has this many nodes or fewer; the root summarises them.
- `TOP_K = 4` – how many nodes a search returns.

In [2]:
DATA_FILE = Path("data/document_a.md")
CACHE_FILE = Path("raptor_tree.json")

LLM_MODEL = os.getenv("RAG_LLM", "ollama:mistral-small3.2:24b")
EMBEDDING_MODEL = "embeddinggemma:300m"

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 0
MIN_CHARS = 80

REDUCED_DIM = 8
MAX_CLUSTERS = 10
CLUSTER_THRESHOLD = 0.1
MAX_CLUSTER_CHARS = 5000
MIN_NODES = 4

TOP_K = 4

print("LLM:", LLM_MODEL)

LLM: ollama:mistral-small3.2:24b


## Step 1 – LOAD

One Markdown file, with two small clean-ups. The export that produced it escaped every dollar sign as `\$`; we undo that, otherwise the backslashes end up in the summaries and in the answers. It also separates the statements with `---` horizontal rules, which carry no content and would otherwise stick to the end of the preceding text; those lines are removed. The heading lines show the structure we are going to chunk along: four statements, then "Notes to Financial Statements" with ten notes, the first of which has its own sub-headings.

In [3]:
text = DATA_FILE.read_text(encoding="utf-8").replace("\\$", "$")
text = "\n".join(line for line in text.splitlines() if line.strip() != "---")
print(f"{len(text):,} characters\n")
for line in text.splitlines():
    if line.startswith("#"):
        print(" ", line)

20,283 characters

  # Financial Statements
  ## Balance Sheets
  ## Statements of Income
  ## Statement of Stockholder's Equity
  ## Statements of Cash Flows (Continued)
  ### Non-cash Transactions
  ## Notes to Financial Statements
  ### Note 1 - Nature of Operations and Summary of Significant Accounting Policies
  #### Nature of Operations
  #### Cash and Cash Equivalents
  #### Accounts Receivable and Credit Policies
  #### Revenue and Unbilled Revenue Receivable
  #### Fixed Assets
  #### Impairment of Long-lived Assets
  #### Income Taxes
  #### Advertising Costs
  #### Use of Estimates
  #### Compensated Absences
  ### Note 2 - Fixed Assets
  ### Note 3 - Lines of Credit and Letter of Credit
  ### Note 4 - Long-term Debt
  ### Note 5 - Capital Leases
  ### Note 6 - Operating Leases
  ### Note 7 - Related Party Transactions
  ### Note 8 - Retirement Plan
  ### Note 9 - Concentrations
  ### Note 10 - Subsequent Events


## Step 2 – CHUNK into leaves

Three steps, because the document has structure worth keeping:

1. `MarkdownHeaderTextSplitter` cuts the text at the `#`–`####` headings. Each piece gets the *path* of headings above it in its metadata (`h1` … `h4`) and the heading lines themselves are removed from the content.
2. Sections longer than `CHUNK_SIZE` are split further by the `RecursiveCharacterTextSplitter` from Example 1, with one extra separator: `(?<=</tr>)`, a lookbehind that splits right after a table row. HTML tables in this file are single lines, so without it a long table would be cut at a space in the middle of a cell. Pieces shorter than `MIN_CHARS` are dropped.
3. Every leaf gets a *breadcrumb* – its heading path, e.g. `Financial Statements > Notes to Financial Statements > Note 4 - Long-term Debt` – prepended to its text. A fragment of a table is then self-describing: the second half of the balance sheet still says "Balance Sheets". (This is what a structure-aware splitter buys you; it also helps the clustering, because pieces of one section embed close together.)

Each leaf is a `Document` with `node_id` (`L0-0`, `L0-1`, …), `layer` 0 and its `section`. The sample at the end shows one leaf in full.

In [4]:
HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4")]
sections = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS).split_text(text)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", r"(?<=</tr>)", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
)
pieces = [p for p in splitter.split_documents(sections) if len(p.page_content) >= MIN_CHARS]


def breadcrumb(metadata):
    return " > ".join(metadata[k] for k in ("h1", "h2", "h3", "h4") if k in metadata)


leaves = [
    Document(
        page_content=f"{breadcrumb(p.metadata)}\n\n{p.page_content}",
        metadata={"node_id": f"L0-{i}", "layer": 0, "section": breadcrumb(p.metadata)},
    )
    for i, p in enumerate(pieces)
]
print(f"{len(sections)} sections -> {len(leaves)} leaves\n")
for leaf in leaves:
    print(f"  {leaf.metadata['node_id']:5} {len(leaf.page_content):5d} chars  {leaf.metadata['section']}")

print("\nExample leaf:\n")
print(next(leaf for leaf in leaves if "Total long-term debt" in leaf.page_content).page_content)

26 sections -> 30 leaves

  L0-0   1221 chars  Financial Statements > Balance Sheets
  L0-1   1161 chars  Financial Statements > Balance Sheets
  L0-2    152 chars  Financial Statements > Balance Sheets
  L0-3    900 chars  Financial Statements > Statements of Income
  L0-4   1076 chars  Financial Statements > Statement of Stockholder's Equity
  L0-5   1063 chars  Financial Statements > Statements of Cash Flows (Continued)
  L0-6    307 chars  Financial Statements > Statements of Cash Flows (Continued) > Non-cash Transactions
  L0-7    775 chars  Financial Statements > Notes to Financial Statements > Note 1 - Nature of Operations and Summary of Significant Accounting Policies > Nature of Operations
  L0-8    476 chars  Financial Statements > Notes to Financial Statements > Note 1 - Nature of Operations and Summary of Significant Accounting Policies > Cash and Cash Equivalents
  L0-9    967 chars  Financial Statements > Notes to Financial Statements > Note 1 - Nature of Operations and S

## The models and the summarisation prompt

`OllamaEmbeddings` and `init_chat_model` are set up as in the earlier examples.

The summary prompt is written for a *search index*, not for a reader, and it does three jobs:

- **Decontextualise.** The first sentence must say what the excerpts cover and, if any excerpt names it, which company they belong to. Most of the document says only "the Company"; a summary that starts with "These excerpts cover the long-term debt of The Employer Group, Inc. …" is self-contained and retrievable. This is the same idea as the propositions of Example 2. The prompt also says what to write when the name is *not* known ("the company") – while developing this notebook the model otherwise wrote "an unnamed company", and that phrase was then copied up the tree and crowded out the real name when a lower summary did contain it.
- **Keep the facts that matter, exactly as stated.** The LLM may later have to answer from the summary alone, so figures, dates, rates and names must survive, and nothing may be invented.
- **Stay short.** Upper layers are *overviews*; the details remain retrievable in the leaves, which stay in the index. A summary as long as its inputs is just a blurrier copy of them, and its embedding is a blend of everything. The limit is called *strict* on purpose, and even so a 24B local model stretches it on dense inputs – while developing this notebook it produced a 900-word "summary" of a 2,000-token cluster. The word limit is the advisory half of the fix; `MAX_CLUSTER_CHARS`, which keeps the inputs small, is the structural half.

There are two prompts, matched to the kind of input. `SUMMARY_PROMPT` is used for layer 1, whose inputs are raw excerpts. `OVERVIEW_PROMPT` is used for every layer above it and for the root, whose inputs are summaries of parts: it asks for an overview that opens with which company the statements belong to and what it does, if any summary says so. That last clause matters: the model tends to anchor on the *first* input, and if that one only says "the company" the name from a later input gets lost; asking for it explicitly fixes that, and it is what an overview should open with anyway.

`summarize(nodes, prompt)` joins the children's texts, calls the model, and prints the token usage of the call. `usage_metadata` is LangChain's provider-independent token count; the input number is a sanity check that the whole cluster reached the model, the output number shows how far the model respected the length limit.

In [5]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = init_chat_model(LLM_MODEL, temperature=0)

SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You summarise excerpts from a company's financial statements for a search index. "
     "Write one paragraph of at most 150 words; this limit is strict, so when the excerpts "
     "hold more facts than fit, keep the totals and the most significant items and leave "
     "the rest out. Start with a sentence that says what the excerpts cover. If any excerpt "
     "names the company, use that name; if none does, write 'the company' and never call it "
     "unnamed. Then give the most important facts - figures, dates, rates, names and "
     "conditions - exactly as stated. Do not use lists and do not add anything that is not "
     "in the excerpts."),
    ("human", "Excerpts:\n\n{context}"),
])

OVERVIEW_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You write an overview of a company's financial statements, or of part of them, for a "
     "search index, from summaries of their sections. Write one paragraph of at most 150 "
     "words; this limit is strict, so keep only the totals and the most significant items. "
     "The first sentence must say which company the statements belong to and what it does, "
     "if any summary says so, what the summaries cover, and which periods they refer to. "
     "Then give the most important figures and topics exactly as stated. Do not use lists "
     "and do not add anything that is not in the summaries."),
    ("human", "Summaries:\n\n{context}"),
])


def summarize(nodes, prompt=SUMMARY_PROMPT):
    context = "\n\n---\n\n".join(n.page_content for n in nodes)
    message = (prompt | llm).invoke({"context": context})
    usage = message.usage_metadata or {}
    print(f"      summarised {len(nodes)} nodes: {usage.get('input_tokens')} tokens in, "
          f"{usage.get('output_tokens')} out")
    return message.content.strip()

## Clustering one layer

`cluster_nodes(vectors)` takes the embeddings of one layer and returns the clusters as lists of indices. Step by step:

1. **Caps for small layers.** With *n* nodes we allow at most `n // 3` clusters (so clusters have three members on average) and at most `n // 3` PCA dimensions. A Gaussian with a full covariance matrix in *d* dimensions has *d(d+1)/2* covariance parameters to estimate, which needs clearly more points than dimensions; without the cap, a layer of eight summaries in eight dimensions would be fitted perfectly and meaninglessly.
2. **PCA** projects the 768-dimensional vectors onto the `dim` directions of largest variance. This is the stand-in for the paper's UMAP: it makes the mixture model tractable and removes directions that are mostly noise.
3. **GMM with BIC.** For every candidate number of clusters *k* we fit a Gaussian Mixture Model (`n_init=5` random restarts, `random_state=0` for reproducibility) and compute its BIC – the model's log-likelihood penalised by its number of parameters. Lower is better. The *k* with the lowest BIC wins, so the number of clusters is chosen by the data, not by us.
4. **Soft membership.** `predict_proba` gives, for every node, its probability of belonging to each cluster. A node joins every cluster whose probability exceeds `CLUSTER_THRESHOLD`. The probabilities sum to 1, so every node is in at least one cluster; a node sitting between two topics can be in two.
5. **Bound the size.** The paper re-clusters any cluster whose text would not fit the summariser's context window. With a handful of nodes a second mixture model is not meaningful, so `split_oversized` simply cuts a cluster above `MAX_CLUSTER_CHARS` in half, in document order, until the parts fit. The purpose is the same: no summary has to compress more than a few thousand characters.

Note that nothing here knows about sections or headings – the grouping is purely geometric, based on where the vectors sit.

One honest limitation: with 30 leaves in 8 dimensions, every cluster holds fewer leaves than there are dimensions. Each Gaussian then collapses onto its few points, and BIC's choice of *k* is driven by that collapse rather than by evidence in the data; near the top of the tree, with a handful of summaries, any statistical model selection is coarse anyway. The clusters are still topically coherent, because the component means behave like k-means centres, but their number should not be read as the number of topics in the document. The paper's own recipe behaves the same way on 30 leaves – notebook 5c shows both in detail. On a corpus of thousands of chunks, the paper's use case, the leaf layer is out of this regime.

In [6]:
def split_oversized(members, texts):
    if len(members) < 2 or sum(len(texts[i]) for i in members) <= MAX_CLUSTER_CHARS:
        return [members]
    half = len(members) // 2
    return split_oversized(members[:half], texts) + split_oversized(members[half:], texts)


def cluster_nodes(vectors, texts):
    n = len(vectors)
    dim = max(1, min(REDUCED_DIM, n // 3))
    max_k = max(1, min(MAX_CLUSTERS, n // 3))

    X = PCA(n_components=dim, random_state=0).fit_transform(np.array(vectors))
    models = [GaussianMixture(k, n_init=5, random_state=0).fit(X) for k in range(1, max_k + 1)]
    best = min(models, key=lambda m: m.bic(X))

    probabilities = best.predict_proba(X)
    clusters = [list(np.where(probabilities[:, c] > CLUSTER_THRESHOLD)[0])
                for c in range(best.n_components)]
    return [part for members in clusters for part in split_oversized(members, texts)]

### A look at the clusters before any LLM call

Embedding the leaves takes a second; the clustering is instant. Rows of the same table and notes on the same topic land together – the balance-sheet pieces, the long-term-debt pieces, the accounting policies – although the algorithm never saw a heading. This is the structure the summaries will follow. If a cluster exceeded `MAX_CLUSTER_CHARS`, `split_oversized` would show up here as two consecutive clusters covering the same sections; with the default settings that only happens in the upper layers, where the texts are summaries.

In [7]:
leaf_vectors = embeddings.embed_documents([leaf.page_content for leaf in leaves])
clusters = cluster_nodes(leaf_vectors, [leaf.page_content for leaf in leaves])

print(f"{len(leaves)} leaves -> {len(clusters)} clusters\n")
for c, members in enumerate(clusters):
    print(f"cluster {c} ({len(members)} leaves)")
    for i in members:
        body = leaves[i].page_content.split("\n\n", 1)[1][:45].replace("\n", " ")
        print(f"    {leaves[i].metadata['node_id']:5} {leaves[i].metadata['section'][:68]:68} {body!r}")

30 leaves -> 8 clusters

cluster 0 (3 leaves)
    L0-13 Financial Statements > Notes to Financial Statements > Note 1 - Natu 'The Company has elected under the Internal Re'
    L0-14 Financial Statements > Notes to Financial Statements > Note 1 - Natu 'The Company assesses the likelihood of uncert'
    L0-27 Financial Statements > Notes to Financial Statements > Note 8 - Reti 'The Company has a 401(k) defined contribution'
cluster 1 (5 leaves)
    L0-3  Financial Statements > Statements of Income                          'Years Ended December 31, 2019 and 2018   <tab'
    L0-4  Financial Statements > Statement of Stockholder's Equity             'Years Ended December 31, 2019 and 2018   <tab'
    L0-5  Financial Statements > Statements of Cash Flows (Continued)          '<table><tr><td></td><td>2019</td><td>2018</td'
    L0-8  Financial Statements > Notes to Financial Statements > Note 1 - Natu 'For purposes of the statements of cash flows,'
    L0-28 Financial Statements > Notes to Fi

## Building the tree

`build_tree(leaves)` is the recursion described above:

- `nodes` collects every node of every layer; `layer_nodes` is the layer currently being clustered.
- While the layer has more than `MIN_NODES` nodes: embed it, cluster it, and stop if that yields a single cluster (then there is nothing left to split). Otherwise summarise every cluster into a new node `L{layer}-{cluster}` that records the `node_id`s of its children – with `SUMMARY_PROMPT` for layer 1, whose inputs are excerpts, and `OVERVIEW_PROMPT` above it, where the inputs are summaries – and make those new nodes the next layer.
- Afterwards, whatever the last layer contains is summarised once more into the `root`, with `OVERVIEW_PROMPT`. Its layer number is one above the last layer.

The depth of the tree is therefore decided by the data: the leaves are re-embedded only at layer 0 (we already have their vectors, but keeping the function self-contained is simpler), and every further layer is embedded fresh because its texts are new.

**Cache.** The LLM calls are the expensive part – a dozen calls, around ten minutes with a 24B local model – so the finished tree is written to `CACHE_FILE` as a plain list of `{"text": ..., "metadata": ...}` records, and later runs load it instead. Delete the file to rebuild.

In [8]:
def build_tree(leaves):
    nodes, layer_nodes, layer = list(leaves), list(leaves), 0
    while len(layer_nodes) > MIN_NODES:
        texts = [n.page_content for n in layer_nodes]
        clusters = cluster_nodes(embeddings.embed_documents(texts), texts)
        print(f"  layer {layer}: {len(layer_nodes)} nodes -> {len(clusters)} clusters "
              f"of sizes {[len(c) for c in clusters]}")
        if len(clusters) == 1:
            break
        layer += 1
        prompt = SUMMARY_PROMPT if layer == 1 else OVERVIEW_PROMPT
        new_nodes = []
        for c, members in enumerate(clusters):
            children = [layer_nodes[i] for i in members]
            new_nodes.append(Document(
                page_content=summarize(children, prompt),
                metadata={"node_id": f"L{layer}-{c}", "layer": layer,
                          "children": [child.metadata["node_id"] for child in children]},
            ))
        nodes += new_nodes
        layer_nodes = new_nodes

    print(f"  layer {layer}: {len(layer_nodes)} nodes -> root")
    root = Document(
        page_content=summarize(layer_nodes, OVERVIEW_PROMPT),
        metadata={"node_id": "root", "layer": layer + 1,
                  "children": [n.metadata["node_id"] for n in layer_nodes]},
    )
    return nodes + [root]


if CACHE_FILE.exists():
    print(f"Using cached tree from {CACHE_FILE.name}")
    nodes = [Document(page_content=r["text"], metadata=r["metadata"])
             for r in json.loads(CACHE_FILE.read_text())]
else:
    nodes = build_tree(leaves)
    CACHE_FILE.write_text(json.dumps(
        [{"text": n.page_content, "metadata": n.metadata} for n in nodes], indent=2))

print(f"\n{len(nodes)} nodes in {max(n.metadata['layer'] for n in nodes) + 1} layers")

  layer 0: 30 nodes -> 8 clusters of sizes [3, 5, 6, 3, 4, 3, 3, 3]
      summarised 3 nodes: 642 tokens in, 170 out
      summarised 5 nodes: 1794 tokens in, 273 out
      summarised 6 nodes: 1505 tokens in, 310 out
      summarised 3 nodes: 952 tokens in, 240 out
      summarised 4 nodes: 685 tokens in, 147 out
      summarised 3 nodes: 1309 tokens in, 286 out
      summarised 3 nodes: 635 tokens in, 143 out
      summarised 3 nodes: 744 tokens in, 220 out
  layer 1: 8 nodes -> 3 clusters of sizes [3, 4, 1]
      summarised 3 nodes: 879 tokens in, 368 out
      summarised 4 nodes: 1019 tokens in, 263 out
      summarised 1 nodes: 269 tokens in, 148 out
  layer 2: 3 nodes -> root
      summarised 3 nodes: 905 tokens in, 487 out

42 nodes in 4 layers


### The finished tree

Top-down: the root, then each layer of summaries with the ids of their children and the first sentence or two of their text. Read a layer-1 summary next to the leaves it was made from (listed in the cluster printout above) to see what the LLM kept.

In [9]:
by_id = {n.metadata["node_id"]: n for n in nodes}
children = {n.metadata["node_id"]: n.metadata.get("children", []) for n in nodes}
top_layer = max(n.metadata["layer"] for n in nodes)

for layer in range(top_layer, 0, -1):
    layer_nodes = [n for n in nodes if n.metadata["layer"] == layer]
    print(f"\n=== layer {layer}: {len(layer_nodes)} node(s) ===")
    for n in layer_nodes:
        print(f"\n{n.metadata['node_id']}  <- children {children[n.metadata['node_id']]}")
        print("   " + n.page_content[:260].replace("\n", " ") + " ...")
print(f"\n=== layer 0: {sum(n.metadata['layer'] == 0 for n in nodes)} leaves ===")


=== layer 3: 1 node(s) ===

root  <- children ['L2-0', 'L2-1', 'L2-2']
   The financial statements of The Employer Group, Inc., a professional employer organization serving small businesses, cover its tax policies, retirement plan, long-term debt, accounting policies, balance sheets, and accounts receivable policies for the periods  ...

=== layer 2: 3 node(s) ===

L2-0  <- children ['L1-0', 'L1-1', 'L1-2']
   The financial statements of a company that provides unspecified services cover its tax policies, retirement plan, and key financial data for the years ended December 31, 2019, and 2018. The company reported gross revenue of $103,015,345 in 2019, up from $95,79 ...

L2-1  <- children ['L1-3', 'L1-4', 'L1-5', 'L1-7']
   This overview summarizes the financial statements of a company, covering its long-term debt, accounting policies, balance sheets, and accounts receivable policies for the periods ending December 31, 2019, and 2018. The company's total long-term debt was $287,0 ...


## Indexing: the collapsed tree

All nodes of all layers go into **one** Chroma collection – that is the collapsed tree. Each node's text is embedded by `embeddings` when it is added, exactly like the chunks in Example 1. The metadata carries `layer` and `node_id`; Chroma metadata must be flat scalars, so the `children` lists stay in the Python dict built above. We pass the node ids as Chroma ids, which lets the traversal below address individual nodes.

`reset_collection()` makes the cell safe to re-run, as in the earlier notebooks.

In [10]:
index_docs = [
    Document(page_content=n.page_content,
             metadata={"node_id": n.metadata["node_id"], "layer": n.metadata["layer"],
                       "section": n.metadata.get("section", "")})
    for n in nodes
]

vector_store = Chroma(
    collection_name="raptor",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
vector_store.reset_collection()
ids = vector_store.add_documents(index_docs, ids=[d.metadata["node_id"] for d in index_docs])
print(f"Indexed {len(ids)} nodes")

Indexed 42 nodes


## Three ways to retrieve

- `leaves_only(question)` – the baseline from Example 1: a similarity search restricted to layer 0. The restriction is a metadata filter, `filter={"layer": 0}`, which LangChain hands to Chroma as a `where` clause.
- `collapsed_tree(question)` – the same search over all layers. Nothing else changes; the query decides whether a leaf or a summary is closest.
- `tree_traversal(question, per_level)` – start with the root as the *frontier*, pick the best `per_level` nodes among the frontier, replace the frontier by their children, repeat until the frontier is empty (the leaves have no children). Each step is again a filtered similarity search, this time with Chroma's `$in` operator: `filter={"node_id": {"$in": frontier}}`. The result contains nodes from every level, roughly `per_level` per layer.

`show` prints rank, cosine similarity, layer and node id, and the beginning of the node text, so the three strategies can be compared side by side.

In [11]:
def leaves_only(question):
    return vector_store.similarity_search_with_relevance_scores(question, k=TOP_K, filter={"layer": 0})


def collapsed_tree(question):
    return vector_store.similarity_search_with_relevance_scores(question, k=TOP_K)


def tree_traversal(question, per_level=2):
    frontier, selected = ["root"], []
    while frontier:
        hits = vector_store.similarity_search_with_relevance_scores(
            question, k=per_level, filter={"node_id": {"$in": frontier}})
        selected += hits
        frontier = [child for doc, _ in hits for child in children[doc.metadata["node_id"]]]
    return selected


def show(label, hits):
    print(f"  {label}:")
    for rank, (doc, score) in enumerate(hits, 1):
        preview = doc.page_content[:75].replace("\n", " ")
        print(f"    {rank}. cos={score:.3f}  [layer {doc.metadata['layer']}  {doc.metadata['node_id']:5}]  {preview}...")

## Run it

Four questions at different granularities:

1. a **detail** – one interest rate in Note 3; the right answer is a single leaf;
2. a **topic** – debt and leases, spread over Notes 3–7 and two tables; the right answer is a layer-1 summary, or several leaves;
3. the **whole document** – how the year went across all statements; the right answer is the root or a layer-2 summary;
4. a question that *sounds* broad but is a single fact stated once, in Note 1 – what business the company is in. It is a test of whether a rare fact survives: the leaf holds it, and the summaries on the leaf's branch should too.

For each question the three retrieval strategies are printed, then the LLM answers from the collapsed-tree hits (the paper's recommendation). Each context item is labelled with its layer so you can see which level the answer came from. Watch the layer column: it should move up from question 1 to question 3.

In [12]:
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the context below. "
     "If the context does not contain the answer, say you don't know. "
     "Cite the context items you used like [1], [2].\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])
answer_chain = PROMPT | llm | StrOutputParser()

QUESTIONS = [
    "What interest rate does the $500,000 revolving credit loan bear?",
    "What debt and lease obligations does the company have?",
    "How did the company's results and financial position change from 2018 to 2019?",
    "What kind of business is the company in?",
]

for question in QUESTIONS:
    print(f"\nQuestion: {question!r}")
    show("Leaves only   ", leaves_only(question))
    hits = collapsed_tree(question)
    show("Collapsed tree", hits)
    show("Tree traversal", tree_traversal(question))

    context = "\n\n".join(
        f"[{i}] (layer {doc.metadata['layer']}, {doc.metadata['node_id']})\n{doc.page_content}"
        for i, (doc, _) in enumerate(hits, 1)
    )
    print("\nAnswer (collapsed tree):", answer_chain.invoke({"context": context, "question": question}))
    print("-" * 80)


Question: 'What interest rate does the $500,000 revolving credit loan bear?'
  Leaves only   :
    1. cos=0.524  [layer 0  L0-19]  Financial Statements > Notes to Financial Statements > Note 3 - Lines of Cr...
    2. cos=0.403  [layer 0  L0-20]  Financial Statements > Notes to Financial Statements > Note 3 - Lines of Cr...
    3. cos=0.353  [layer 0  L0-21]  Financial Statements > Notes to Financial Statements > Note 4 - Long-term D...
    4. cos=0.340  [layer 0  L0-24]  Financial Statements > Notes to Financial Statements > Note 5 - Capital Lea...
  Collapsed tree:
    1. cos=0.524  [layer 0  L0-19]  Financial Statements > Notes to Financial Statements > Note 3 - Lines of Cr...
    2. cos=0.473  [layer 1  L1-7 ]  These excerpts cover the company's accounts receivable policies, lines of c...
    3. cos=0.403  [layer 0  L0-20]  Financial Statements > Notes to Financial Statements > Note 3 - Lines of Cr...
    4. cos=0.359  [layer 2  L2-1 ]  This overview summarizes the financial statem

## Things to try

1. **Index the summary, return the full text.** When a summary node is retrieved, replace it by the concatenated text of its leaves (follow `children` down to layer 0) before building the context – the first technique in the table at the top. Compare the answers to question 2: does the LLM get more precise, or does the context become too long?
2. Lower `CLUSTER_THRESHOLD` to 0.02, delete `raptor_tree.json` and rebuild. Which leaves now sit in two clusters, and do their summaries overlap?
3. Set `MIN_NODES = 10` for a flatter tree (leaves → one layer of summaries → root). Does the collapsed-tree retrieval change?
4. Ask a question that needs a detail *and* its context, e.g. "How much of the long-term debt is due in 2020, and what share of total long-term debt is that?" Which layers answer it?
5. Add the three `.txt` documents from the earlier examples to the leaves. The tree then spans a corpus, as in the paper, and the root should describe all four documents.
6. Replace the PCA step by UMAP, the paper's choice: `umap.UMAP(n_neighbors=int((n - 1) ** 0.5), n_components=dim, metric="cosine").fit_transform(...)` (notebook 5a explains the parameters). Do the clusters change? Notebook 5c, section 7, runs the paper's complete clustering recipe on the same leaves.